In [1]:
import numpy as np
import pandas as pd
import copy
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    confusion_matrix, matthews_corrcoef
)
RANDOM_SEED = 42
N_FOLDS     = 5

TRAIN_PATH = r"C:\Users\Namit\DIA\drug_induced_autoimmunity_prediction\DIA_trainingset_RDKit_descriptors.csv"
TEST_PATH  = r"C:\Users\Namit\DIA\drug_induced_autoimmunity_prediction\DIA_testset_RDKit_descriptors.csv"

np.random.seed(RANDOM_SEED)



In [2]:
# ── Loading data ──
df_train = pd.read_csv(TRAIN_PATH)
df_test  = pd.read_csv(TEST_PATH)

drop_cols    = ['SMILES', 'Label']
X_train_full = df_train.drop(columns=[c for c in drop_cols
                                       if c in df_train.columns])
y_train_full = df_train['Label'].values
X_test_full  = df_test.drop(columns=[c for c in drop_cols
                                      if c in df_test.columns])
y_test_full  = df_test['Label'].values

shared_cols  = [c for c in X_train_full.columns
                if c in X_test_full.columns]
X_train_df   = pd.DataFrame(X_train_full[shared_cols].values,
                             columns=shared_cols)
X_test_df    = pd.DataFrame(X_test_full[shared_cols].values,
                             columns=shared_cols)

print(f"Train: {X_train_df.shape}")
print(f"Test : {X_test_df.shape}")
print(f"Train class dist: {np.bincount(y_train_full.astype(int))}")
print(f"Test  class dist: {np.bincount(y_test_full.astype(int))}")
print(f"Features used   : ALL {len(shared_cols)} RDKit descriptors")



Train: (477, 196)
Test : (120, 196)
Train class dist: [359 118]
Test  class dist: [90 30]
Features used   : ALL 196 RDKit descriptors


In [3]:
# ── Metrics function ──
def compute_metrics(y_true, y_pred, y_prob, model_name=""):
    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0,1]
    ).ravel()
    acc = accuracy_score(y_true, y_pred)
    sen = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spe = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1  = f1_score(y_true, y_pred, zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred)
    try:
        auc = roc_auc_score(y_true, y_prob)
    except Exception:
        auc = 0.5
    return {
        'Model':       model_name,
        'Accuracy':    round(acc, 4),
        'Sensitivity': round(sen, 4),
        'Specificity': round(spe, 4),
        'F1':          round(f1,  4),
        'AUC':         round(auc, 4),
        'MCC':         round(mcc, 4),
        'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn
    }



In [4]:
# ── Models ──
MODELS_CLS = {
    'LogReg':  LogisticRegression(
                   max_iter=1000, random_state=RANDOM_SEED),
    'SVM-RBF': SVC(
                   kernel='rbf', probability=True,
                   C=10, random_state=RANDOM_SEED),
    'RF':      RandomForestClassifier(
                   n_estimators=200, random_state=RANDOM_SEED,
                   n_jobs=-1),
    'XGBoost': XGBClassifier(
                   n_estimators=200, random_state=RANDOM_SEED,
                   use_label_encoder=False,
                   eval_metric='logloss', verbosity=0)
}

# ── 5-Fold CV ─────────────────────────────────────────────────────────────────
skf = StratifiedKFold(
    n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED
)

all_results    = []
cls_fold_probs = {m: {} for m in MODELS_CLS.keys()}

print(f"\nRunning 5-fold CV on ALL {len(shared_cols)} features...\n")
print("=" * 70)

for fold, (tr_idx, _) in enumerate(
        skf.split(X_train_df, y_train_full)):

    print(f"\nFold {fold+1}/{N_FOLDS}")

    X_fold_tr = X_train_df.iloc[tr_idx].reset_index(drop=True)
    y_fold_tr = y_train_full[tr_idx]

    # SMOTE inside fold
    smote = SMOTE(random_state=RANDOM_SEED + fold)
    X_sm_f, y_sm_f = smote.fit_resample(X_fold_tr, y_fold_tr)

    print(f"  After SMOTE: {X_sm_f.shape} "
          f"({np.bincount(y_sm_f.astype(int))})")

    # Scale inside fold
    scaler   = StandardScaler()
    X_sm_sc  = scaler.fit_transform(X_sm_f)
    X_te_sc  = scaler.transform(X_test_df.values)

    # Train each model
    for model_name, model_proto in MODELS_CLS.items():
        model = copy.deepcopy(model_proto)
        model.fit(X_sm_sc, y_sm_f)

        y_pred = model.predict(X_te_sc)
        y_prob = model.predict_proba(X_te_sc)[:, 1]

        cls_fold_probs[model_name][fold] = {
            'y_pred': y_pred,
            'y_prob': y_prob
        }

        metrics = compute_metrics(
            y_test_full, y_pred, y_prob,
            model_name=model_name
        )
        metrics['Fold']     = fold + 1
        metrics['Features'] = 'All-196'
        all_results.append(metrics)

        print(f"  {model_name:<10} "
              f"AUC={metrics['AUC']:.4f}  "
              f"SEN={metrics['Sensitivity']:.4f}  "
              f"SPE={metrics['Specificity']:.4f}")

print("\n All folds complete")


Running 5-fold CV on ALL 196 features...


Fold 1/5
  After SMOTE: (574, 196) ([287 287])
  LogReg     AUC=0.7533  SEN=0.6000  SPE=0.8556
  SVM-RBF    AUC=0.8089  SEN=0.4000  SPE=0.9667
  RF         AUC=0.8591  SEN=0.4333  SPE=0.9556
  XGBoost    AUC=0.8607  SEN=0.4333  SPE=0.9111

Fold 2/5
  After SMOTE: (574, 196) ([287 287])
  LogReg     AUC=0.7085  SEN=0.5000  SPE=0.7444
  SVM-RBF    AUC=0.8311  SEN=0.5333  SPE=0.9444
  RF         AUC=0.8659  SEN=0.4667  SPE=0.9667
  XGBoost    AUC=0.8759  SEN=0.6667  SPE=0.9444

Fold 3/5
  After SMOTE: (574, 196) ([287 287])
  LogReg     AUC=0.7030  SEN=0.4667  SPE=0.7667
  SVM-RBF    AUC=0.7678  SEN=0.3667  SPE=0.9333
  RF         AUC=0.8607  SEN=0.4333  SPE=0.9556
  XGBoost    AUC=0.8622  SEN=0.4667  SPE=0.9222

Fold 4/5
  After SMOTE: (574, 196) ([287 287])
  LogReg     AUC=0.7519  SEN=0.5667  SPE=0.8333
  SVM-RBF    AUC=0.7833  SEN=0.4000  SPE=0.9556
  RF         AUC=0.8574  SEN=0.3000  SPE=0.9444
  XGBoost    AUC=0.8289  SEN=0.3000  SPE=0.93

In [5]:
# ── Aggregate results ─────────────────────────────────────────────────────────
metrics_cols = ['Accuracy','Sensitivity','Specificity','F1','AUC','MCC']

df_cls_results = pd.DataFrame(all_results)

df_cls_agg = (
    df_cls_results
    .groupby('Model')[metrics_cols]
    .agg(['mean','std'])
    .round(4)
)
df_cls_agg.columns = [f"{m}_{s}" for m, s in df_cls_agg.columns]
df_cls_agg = df_cls_agg.reset_index()

# ── Print results ─────────────────────────────────────────────────────────────
print("\n CLASSICAL MODELS — ALL 196 FEATURES (Mean ± Std, 5-fold CV)\n")
print("=" * 110)
header = f"  {'Model':<12}" + "".join(
    [f"  {m:>18}" for m in metrics_cols]
)
print(header)
print("  " + "-" * (12 + 20*len(metrics_cols)))

for model in ['LogReg','SVM-RBF','RF','XGBoost']:
    row = df_cls_agg[df_cls_agg['Model']==model]
    if len(row) == 0:
        continue
    row  = row.iloc[0]
    line = f"  {model:<12}"
    for m in metrics_cols:
        line += f"  {row[f'{m}_mean']:.4f}±{row[f'{m}_std']:.4f}  "
    print(line)




📊 CLASSICAL MODELS — ALL 196 FEATURES (Mean ± Std, 5-fold CV)

  Model                   Accuracy         Sensitivity         Specificity                  F1                 AUC                 MCC
  ------------------------------------------------------------------------------------------------------------------------------------
  LogReg        0.7333±0.0468    0.5400±0.0548    0.7978±0.0461    0.5052±0.0686    0.7256±0.0249    0.3256±0.1005  
  SVM-RBF       0.8150±0.0199    0.4133±0.0691    0.9489±0.0127    0.5258±0.0632    0.7984±0.0242    0.4493±0.0684  
  RF            0.8183±0.0216    0.4000±0.0667    0.9578±0.0093    0.5222±0.0718    0.8617±0.0037    0.4566±0.0777  
  XGBoost       0.8050±0.0415    0.4400±0.1442    0.9266±0.0127    0.5223±0.1296    0.8438±0.0341    0.4220±0.1402  


In [7]:
# ── Balance score ─────────────────────────────────────────────────────────────
print("\n BALANCE SCORES (1 - |SEN - SPE|)\n")
print("=" * 50)
for model in ['LogReg','SVM-RBF','RF','XGBoost']:
    row = df_cls_agg[df_cls_agg['Model']==model].iloc[0]
    bal = 1 - abs(row['Sensitivity_mean'] - row['Specificity_mean'])
    print(f"  {model:<12} Balance={bal:.4f}  "
          f"SEN={row['Sensitivity_mean']:.4f}  "
          f"SPE={row['Specificity_mean']:.4f}")

# ── Save ──────────────────────────────────────────────────────────────────────
import pickle, os
df_cls_results.to_csv(r'C:\Users\Namit\DIA\Classical models\classical_all_features_raw.csv', index=False)
df_cls_agg.to_csv(r'C:\Users\Namit\DIA\Classical models\classical_all_features_agg.csv', index=False)

with open(r'C:\Users\Namit\DIA\Classical models\cls_fold_probs_all_features.pkl', 'wb') as f:
    pickle.dump(cls_fold_probs, f)

print("\n Results saved:")
print("   classical_all_features_raw.csv")
print("   classical_all_features_agg.csv")
print("   cls_fold_probs_all_features.pkl")



📊 BALANCE SCORES (1 - |SEN - SPE|)

  LogReg       Balance=0.7422  SEN=0.5400  SPE=0.7978
  SVM-RBF      Balance=0.4644  SEN=0.4133  SPE=0.9489
  RF           Balance=0.4422  SEN=0.4000  SPE=0.9578
  XGBoost      Balance=0.5134  SEN=0.4400  SPE=0.9266

✅ Results saved:
   classical_all_features_raw.csv
   classical_all_features_agg.csv
   cls_fold_probs_all_features.pkl


In [8]:
print("\n CLASSICAL MODELS — ALL 196 FEATURES (Mean ± Std, 5-fold CV)\n")
print("=" * 110)
header = f"  {'Model':<12}" + "".join(
    [f"  {m:>18}" for m in metrics_cols]
)
print(header)
print("  " + "-" * (12 + 20*len(metrics_cols)))

for model in ['LogReg','SVM-RBF','RF','XGBoost']:
    row = df_cls_agg[df_cls_agg['Model']==model]
    if len(row) == 0:
        continue
    row  = row.iloc[0]
    line = f"  {model:<12}"
    for m in metrics_cols:
        line += f"  {row[f'{m}_mean']:.4f}±{row[f'{m}_std']:.4f}  "
    print(line)

print("\n BALANCE SCORES (1 - |SEN - SPE|)\n")
print("=" * 50)
for model in ['LogReg','SVM-RBF','RF','XGBoost']:
    row = df_cls_agg[df_cls_agg['Model']==model].iloc[0]
    bal = 1 - abs(row['Sensitivity_mean'] - row['Specificity_mean'])
    print(f"  {model:<12} "
          f"AUC={row['AUC_mean']:.4f}  "
          f"ACC={row['Accuracy_mean']:.4f}  "
          f"SEN={row['Sensitivity_mean']:.4f}  "
          f"SPE={row['Specificity_mean']:.4f}  "
          f"F1={row['F1_mean']:.4f}  "
          f"MCC={row['MCC_mean']:.4f}  "
          f"Balance={bal:.4f}")


📊 CLASSICAL MODELS — ALL 196 FEATURES (Mean ± Std, 5-fold CV)

  Model                   Accuracy         Sensitivity         Specificity                  F1                 AUC                 MCC
  ------------------------------------------------------------------------------------------------------------------------------------
  LogReg        0.7333±0.0468    0.5400±0.0548    0.7978±0.0461    0.5052±0.0686    0.7256±0.0249    0.3256±0.1005  
  SVM-RBF       0.8150±0.0199    0.4133±0.0691    0.9489±0.0127    0.5258±0.0632    0.7984±0.0242    0.4493±0.0684  
  RF            0.8183±0.0216    0.4000±0.0667    0.9578±0.0093    0.5222±0.0718    0.8617±0.0037    0.4566±0.0777  
  XGBoost       0.8050±0.0415    0.4400±0.1442    0.9266±0.0127    0.5223±0.1296    0.8438±0.0341    0.4220±0.1402  


📊 BALANCE SCORES (1 - |SEN - SPE|)

  LogReg       AUC=0.7256  ACC=0.7333  SEN=0.5400  SPE=0.7978  F1=0.5052  MCC=0.3256  Balance=0.7422
  SVM-RBF      AUC=0.7984  ACC=0.8150  SEN=0.4133  SPE=0.94

In [10]:
import pandas as pd

# ── Load from saved CSV ───────────────────────────────────────────────────────
df_cls_agg = pd.read_csv(
    r'C:\Users\Namit\DIA\Classical models\classical_all_features_agg.csv'
)

# ── Select only the 4 metrics needed ─────────────────────────────────────────
metrics_needed = ['Accuracy', 'AUC', 'Sensitivity', 'Specificity']

print("CLASSICAL MODELS — ALL 196 FEATURES\n")
print("    (Mean ± Std across 5 folds, evaluated on held-out test set)\n")
print("=" * 75)
print(f"  {'Model':<12}", end="")
for m in metrics_needed:
    print(f"  {m:>18}", end="")
print()
print("  " + "-" * 72)

for model in ['LogReg', 'SVM-RBF', 'RF', 'XGBoost']:
    row = df_cls_agg[df_cls_agg['Model'] == model]
    if len(row) == 0:
        continue
    row  = row.iloc[0]
    line = f"  {model:<12}"
    for m in metrics_needed:
        mu  = row[f'{m}_mean']
        std = row[f'{m}_std']
        line += f"  {mu:.4f}±{std:.4f}  "
    print(line)

print("\n")

📊 CLASSICAL MODELS — ALL 196 FEATURES

    (Mean ± Std across 5 folds, evaluated on held-out test set)

  Model                   Accuracy                 AUC         Sensitivity         Specificity
  ------------------------------------------------------------------------
  LogReg        0.7333±0.0468    0.7256±0.0249    0.5400±0.0548    0.7978±0.0461  
  SVM-RBF       0.8150±0.0199    0.7984±0.0242    0.4133±0.0691    0.9489±0.0127  
  RF            0.8183±0.0216    0.8617±0.0037    0.4000±0.0667    0.9578±0.0093  
  XGBoost       0.8050±0.0415    0.8438±0.0341    0.4400±0.1442    0.9266±0.0127  


